# 07. Зведене порівняння всіх моделей

Читає `metrics.json` і `history.csv` усіх запусків з `hw2_runs/<модель>/<запуск>/`, нічого не перенавчає. Графіки зберігаються в `images/` для звіту.

Результати Faster R-CNN треба покласти в `hw2_runs/faster_rcnn/<запуск>/` (вихід kernel-ів Соломії).

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

RUNS = Path.cwd().parent.parent / "hw2_runs"
IMAGES = Path.cwd().parent / "images"
MODELS = ["faster_rcnn", "retinanet", "yolo", "rtdetr"]
COLS = ["mAP", "AP50", "AP75", "AP_small", "AP_medium", "AP_large", "mAP_test_like"]

## 1. Усі запуски

In [ ]:
rows = {}
for path in sorted(RUNS.glob("*/*/metrics.json")):
    model, run = path.parent.parent.name, path.parent.name
    if run.startswith("smoke") or model not in MODELS:
        continue
    m = json.loads(path.read_text())
    rows[(model, run)] = {**{c: m["val"].get(c) for c in COLS}, "ms/кадр": m.get("ms_per_image")}
runs = pd.DataFrame(rows).T.reindex(MODELS, level=0)
runs.to_csv(RUNS / "all_runs.csv")
runs.astype(float).round(4)

## 2. Найкращий запуск кожної моделі і фінальний пайплайн

In [ ]:
best = runs.astype(float).groupby(level=0).apply(lambda g: g.loc[g["mAP"].idxmax()]).reindex(MODELS).dropna(how="all")
final_path = RUNS / "pipeline" / "final" / "metrics.json"
if final_path.exists():
    final = json.loads(final_path.read_text())
    best.loc[f"pipeline ({final['model']}, {final['variant']})"] = {c: final["val"].get(c) for c in COLS}
best.round(4)

In [ ]:
ax = best[["AP_small", "AP_medium", "AP_large"]].plot.bar(figsize=(9, 4), rot=15, title="AP за розміром об'єктів (найкращий запуск кожної моделі)")
ax.set_ylabel("AP")
plt.tight_layout()
plt.savefig(IMAGES / "ap_by_size.png", dpi=120)
plt.show()

## 3. Val mAP за час навчання: RT-DETR проти YOLO (той самий бюджет)

Обидві криві — mAP Ultralytics на кожному 5-му кадрі val після кожної епохи.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for model, run in [("yolo", "baseline"), ("rtdetr", "baseline"), ("rtdetr", "exp1_queries500"), ("rtdetr", "exp2_giou")]:
    path = RUNS / model / run / "history.csv"
    if path.exists():
        h = pd.read_csv(path)
        ax.plot(h["epoch"], h["mAP"], marker="o", label=f"{model} {run}")
ax.set(xlabel="епоха", ylabel="mAP@[.5:.95] (val_sub)", title="Збіжність: query-based vs CNN")
ax.legend()
plt.tight_layout()
plt.savefig(IMAGES / "rtdetr_vs_yolo.png", dpi=120)
plt.show()

## 4. Val mAP за епохами: torchvision-моделі

Faster R-CNN і RetinaNet, оцінка `hw2_eval` на кожному 5-му кадрі val.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for model in ("faster_rcnn", "retinanet"):
    for path in sorted((RUNS / model).glob("*/history.csv")):
        if path.parent.name.startswith("smoke"):
            continue
        h = pd.read_csv(path)
        ax.plot(h["epoch"], h["mAP"], marker="o", label=f"{model} {path.parent.name}")
ax.set(xlabel="епоха", ylabel="mAP@[.5:.95] (val_sub)", title="Two-stage vs one-stage anchor-based")
ax.legend()
plt.tight_layout()
plt.savefig(IMAGES / "frcnn_vs_retinanet.png", dpi=120)
plt.show()

## 5. Пайплайн: внесок інференсу і моделі

In [ ]:
path = RUNS / "pipeline" / "comparison.csv"
pd.read_csv(path, header=[0, 1], index_col=0).round(4) if path.exists() else "ще немає"